In [1]:
!python -V

Python 3.9.25


In [1]:
import os

print(os.getcwd())
print(os.path.abspath("mlflow.db"))

/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking
/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking/mlflow.db


# Q3. Train a model with autolog

In [3]:
import os
import pickle
import click
import mlflow
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("randomforest")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
def run_train(data_path: str):
    mlflow.sklearn.autolog()
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    with mlflow.start_run():

        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))


if __name__ == '__main__':
    run_train.callback(data_path="./output")

# Q5. Tune model hyperparameters

In [7]:
import os
import pickle
import click
import mlflow
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("hyperparameters")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
@click.option(
    "--num_trials",
    default=15,
    help="The number of parameter evaluations for the optimizer to explore"
)
def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):

        with mlflow.start_run():
            mlflow.log_params(params)
            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            mlflow.log_metric("rmse", rmse)

        return {'loss': rmse, 'status': STATUS_OK}

    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(42)  # for reproducible results
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
    )


if __name__ == '__main__':
    run_optimization.callback(data_path="./output", num_trials=15)

  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

🏃 View run learned-gnu-560 at: http://127.0.0.1:5000/#/experiments/2/runs/4df6fd2521704a24bfd22d34f7bec8e4

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

🏃 View run dashing-trout-593 at: http://127.0.0.1:5000/#/experiments/2/runs/b77b966ba4c64bfca519452e56da07f8

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run debonair-trout-857 at: http://127.0.0.1:5000/#/experiments/2/runs/ac9606b912c041bcbf44f13c1a56e8d9

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run bedecked-mink-755 at: http://127.0.0.1:5000/#/experiments/2/runs/9fef3c8ea713476da1e7fbad4430fa6b

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run rumbling-perch-702 at: http://127.0.0.1:5000/#/experiments/2/runs/f8e06e316cf244bb8a72975eee55d318

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run intrigued-goat-679 at: http://127.0.0.1:5000/#/experiments